# Exploratory Data Analysis of the PLAID Dataset (Submetered Data)
### Appliance Identification from Voltage and Current Waveforms
**EC5203 Machine Learning · Group 6 – Team Plug Detectives**

   **Objective:** Understand the PLAID submetered recordings before feature extraction: what is in the data, what is wrong with it, and what that means for a classifier tested on **unseen homes**.

   **Dataset:** PLAID: 1,876 submetered recordings of 16 appliance types from 56 locations, with current and voltage sampled at 30 kHz (Medico et al., 2020, *Scientific Data* 7:49).

   **Output:** A cleaning plan (Section 13) and `clean_index.csv` (1,810 recordings, 11 types), the input to `02_feature_extraction.ipynb`.

## 1. Setup and Data Paths

In [ ]:
import json
import re
import hashlib
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

try:
    from tqdm.auto import tqdm          # progress bar
except ImportError:
    tqdm = lambda x: x

DATA  = Path.home() / "Documents" / "10084619"
WAVES = DATA / "submetered_new"
AGG   = DATA / "aggregated"
FS    = 30_000
SPC = FS // 60                     # 500 samples per mains cycle
STEADY_S = 20 * SPC / FS           # 0.333 s steady-state window
UNITS = {"w": 1, "watt": 1, "watts": 1, "kw": 1000, "": 1}
NAME_MAP = {"Iron solder": "Soldering Iron"}          # match the submetered label


def load_wave(rec_id):
    return pd.read_csv(WAVES / f"{rec_id}.csv", header=None, names=["current", "voltage"])

def rising_crossings(v, min_gap=int(0.8 * SPC)):
    """Upward zero crossings, ignoring noise wiggles closer than 0.8 cycle apart."""
    idx = np.flatnonzero((v[:-1] < 0) & (v[1:] >= 0))
    keep = []
    for k in idx:
        if not keep or k - keep[-1] >= min_gap:
            keep.append(k)
    return np.array(keep)

def last_cycle(w):
    v, i = w["voltage"].to_numpy(), w["current"].to_numpy()
    r = rising_crossings(v)
    start = r[r <= len(v) - SPC][-1]
    return i[start:start + SPC], v[start:start + SPC]

with open(DATA / "metadata_submetered.json") as f:
    meta_raw = json.load(f)

print("Records:", len(meta_raw))

rec_id = "1"                         # try "1", "1005", "1587"
print(json.dumps(meta_raw[rec_id], indent=2))

FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\User\\Documents\\10084619\\metadata_submetered.json'